In [1]:
import importlib.util
import sys
import pandas as pd
from pathlib import Path
from sqlalchemy import text

In [2]:
def import_module_from_path(module_name: str, file_path: str):
    """Imports a module from a specific file path safely."""
    if not Path(file_path).exists():
        raise FileNotFoundError(f"The file {file_path} does not exist.")

    spec = importlib.util.spec_from_file_location(module_name, Path(file_path))
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

# --- Usage ---

# 1. Define the path once
HELPERS_PATH = "/Users/admin/Nextcloud/08_Accompanying Research/08_08_Data_Extraction/src/helpers.py"

# 2. Import the helper module once
helpers = import_module_from_path("helpers", HELPERS_PATH)

repos_list = helpers.preprocess_repos_list_for_postgres(
    "../data/proc/propensity_match_controls.txt"
)

# 3. Access your functions directly from the module object
engine = helpers.create_postgres_engine(location = "local")

ecosystems_user:cUEqH44AjM4XqXC!tqFM


In [3]:
repos_list.drop_duplicates(subset = "sha_id", inplace = True)
print(f"Unique repositories after dropping duplicates: {len(repos_list)}")

Unique repositories after dropping duplicates: 84


In [4]:
from sqlalchemy import text

# 1. Upload to staging as before
repos_list.to_sql("temp_repo_names", engine, if_exists="replace", index=False)

# 2. Define the UPSERT separately
upsert_sql = """
INSERT INTO repo_names (sha_id, url, owner, repo, host, repo_group_id, added_at)
SELECT sha_id, url, owner, repo, host, repo_group_id, added_at 
FROM temp_repo_names
ON CONFLICT (sha_id) 
DO UPDATE SET 
    url = EXCLUDED.url,
    owner = EXCLUDED.owner,
    repo = EXCLUDED.repo,
    host = EXCLUDED.host,
    repo_group_id = EXCLUDED.repo_group_id,
    added_at = EXCLUDED.added_at;
"""

with engine.begin() as conn:
    # Execute upsert and capture result
    result = conn.execute(text(upsert_sql))
    rows_affected = result.rowcount
    
    # Clean up staging
    conn.execute(text("DROP TABLE temp_repo_names;"))

print(f"Success! Total rows processed (inserted or updated): {rows_affected}")

Success! Total rows processed (inserted or updated): 84
